# Activation Patching With Substructure Probe Directions

In [ ]:
from pathlib import Path
import os, sys

# Allow running from repo root or from notebooks/.
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))


## Experiment
Train row/column/box candidate probes across all layers, then patch selected probe directions into a chosen residual-stream position and compare original vs patched next-token probabilities.


# Activation Patching

Trains substructure probes (row/col/box × digit) on the model's residual stream activations
for **all layers**, then provides an interactive widget to:

1. Browse puzzle sequences and board states
2. Select up to three probe directions (substructure × digit) with configurable coefficients
3. Apply the patch to a chosen layer and compare original vs patched next-token probabilities


In [ ]:
CACHE_PATH = "results/3M-backtracking-packing/activations.npz"
CKPT_DIR   = "results/3M-backtracking-packing/checkpoint"
ACT_TYPE   = "post_mlp"   # or "post_attn"

# Probe training anchor — two modes:
#   "step"     → train at step PROBE_N from the SEP/end-clues token (0 = the SEP token itself)
#   "n_filled" → train at the first position where exactly PROBE_N cells are filled
PROBE_MODE = "step"
PROBE_N    = 0

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button, TextBox, RadioButtons
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from tqdm.auto import tqdm

plt.rcParams['font.family'] = ['Avenir']

from sudoku.probes.session import ProbeSession
from sudoku.activations import load_checkpoint
from sudoku.data import decode_fill, SEP_TOKEN, PAD_TOKEN
from sudoku.data_bt import PUSH_TOKEN, POP_TOKEN, SUCCESS_TOKEN, END_CLUES_TOKEN, PAD_TOKEN_BT
from sudoku.probes.activations import build_grid_at_step

session     = ProbeSession(CACHE_PATH, act_type=ACT_TYPE)
params, model_inst = load_checkpoint(CKPT_DIR)

n_layers    = model_inst.config.n_layers
d_model     = model_inst.config.d_model
vocab_size  = model_inst.config.vocab_size
max_seq_len = model_inst.config.max_seq_len
n_puzzles   = session.n_puzzles

print(f"Model: {n_layers} layers, d_model={d_model}, vocab_size={vocab_size}, max_seq_len={max_seq_len}")
print(f"Dataset: {n_puzzles} puzzles")

In [ ]:
from scripts.analysis.probe_experiments import build_structure_targets
from scripts.analysis.sudoku_state import sequence_length, token_label as get_token_label
from scripts.analysis.plotting import draw_board


def get_seq_len(puzzle_idx: int) -> int:
    return sequence_length(session.sequences[puzzle_idx])


def decode_sequence_labels(puzzle_idx: int) -> list:
    seq = session.sequences[puzzle_idx]
    n = get_seq_len(puzzle_idx)
    return [f"{i:3d}: {get_token_label(int(seq[i]))}" for i in range(n)]


# ── Scrollable token list ─────────────────────────────────────────────────────
class ScrollableTokenList:
    def __init__(self, ax, on_select):
        self.ax            = ax
        self.on_select     = on_select
        self.labels        = []
        self.selected_idx  = 0
        self.view_offset   = 0
        self.visible_count = 38
        ax.set_xticks([])
        ax.set_xlim(0, 1)
        ax.set_yticks([])
        ax.set_ylim(0, 1)
        ax.set_title("Tokens (scroll & click)", fontsize=9)
        self._cid_scroll = ax.figure.canvas.mpl_connect("scroll_event",       self._on_scroll)
        self._cid_click  = ax.figure.canvas.mpl_connect("button_press_event", self._on_click)

    def set_labels(self, labels, selected_idx=0):
        self.labels       = labels
        self.selected_idx = selected_idx
        half = self.visible_count // 2
        self.view_offset = max(0, min(selected_idx - half,
                                      max(0, len(labels) - self.visible_count)))
        self._render()

    def _render(self):
        self.ax.clear()
        self.ax.set_xticks([])
        self.ax.set_xlim(0, 1)
        self.ax.set_yticks([])
        self.ax.set_ylim(0, 1)
        self.ax.set_title("Tokens (scroll & click)", fontsize=9)
        lo, hi = self.view_offset, self.view_offset + self.visible_count
        self.ax.set_ylim(hi, lo)
        n = len(self.labels)
        if 0 <= self.selected_idx < n:
            self.ax.axhspan(self.selected_idx, self.selected_idx + 1,
                            color="#ffe680", alpha=0.6)
        for i, label in enumerate(self.labels):
            if lo - 3 <= i <= hi + 3:
                color  = "red"  if i == self.selected_idx else "black"
                weight = "bold" if i == self.selected_idx else "normal"
                self.ax.text(0.05, i + 0.5, label, transform=self.ax.transData,
                             va="center", color=color, fontweight=weight, fontsize=8)
        self.ax.figure.canvas.draw_idle()

    def _on_scroll(self, event):
        if event.inaxes != self.ax:
            return
        step = -3 if event.button == "up" else 3
        old  = self.view_offset
        self.view_offset = int(np.clip(
            self.view_offset + step, 0,
            max(0, len(self.labels) - self.visible_count)
        ))
        if old != self.view_offset:
            self._render()

    def _on_click(self, event):
        if event.inaxes != self.ax or event.ydata is None:
            return
        idx = int(event.ydata)
        if 0 <= idx < len(self.labels):
            self.selected_idx = idx
            self.on_select(idx)
            self._render()

In [ ]:
if PROBE_MODE == "step":
    probe_idx   = session.index.at_step(PROBE_N).first_per_puzzle()
    anchor_desc = f"step={PROBE_N}"
else:
    probe_idx   = session.index.where_filled(PROBE_N).first_per_puzzle()
    anchor_desc = f"n_filled={PROBE_N}"

train_mask, _test_mask = session.split(probe_idx)
grids_all   = session.grids(probe_idx)
grids_train = [grids_all[i] for i in np.where(train_mask)[0]]

SUBTYPES_LIST = (
    [("row", i) for i in range(9)]
    + [("col", i) for i in range(9)]
    + [("box", i) for i in range(9)]
)

# probes[layer][(subtype, sidx)]            -> fitted MultiOutputClassifier
# probe_vecs[layer][(subtype, sidx, digit)] -> coef_ array, shape (d_model,)
probes     = {}
probe_vecs = {}

for layer in tqdm(range(n_layers), desc="Layer"):
    X       = session.acts(probe_idx, layer=layer)
    X_train = X[train_mask]
    probes[layer]     = {}
    probe_vecs[layer] = {}
    for subtype, sidx in tqdm(SUBTYPES_LIST, desc=f"  L{layer}", leave=False):
        y_tr = build_structure_targets(grids_train, subtype, sidx)
        clf  = MultiOutputClassifier(LogisticRegression(C=1.0, max_iter=500))
        clf.fit(X_train, y_tr.astype(int))
        probes[layer][(subtype, sidx)] = clf
        for digit in range(9):
            probe_vecs[layer][(subtype, sidx, digit)] = clf.estimators_[digit].coef_[0]

print(f"Trained {n_layers} layers x {len(SUBTYPES_LIST)} probes at anchor: {anchor_desc}")

In [ ]:
# Unpatched forward pass
_fwd_clean = jax.jit(lambda seq: model_inst.apply({"params": params}, seq))

# One patched forward pass per layer.
# layer_idx is a Python int in the closure — static at JAX trace time,
# so control flow in the model loop is resolved correctly.
def _make_patched_fn(layer_idx: int):
    @jax.jit
    def fn(seq_arr, pos, delta):
        patch = {"layer": layer_idx, "pos": pos, "delta": delta}
        return model_inst.apply({"params": params}, seq_arr, patch=patch)
    return fn

_fwd_patched = {l: _make_patched_fn(l) for l in range(n_layers)}
print("JIT functions defined (compiled on first call per layer)")

In [ ]:
def run_patched_forward(puzzle_idx: int, seq_pos: int,
                        patch_layer: int, patch_specs: list) -> tuple:
    """Apply a linear combination of probe directions to the residual stream.

    Parameters
    ----------
    puzzle_idx  : index into session.sequences
    seq_pos     : token position to patch and from which to read logits
    patch_layer : transformer block whose post-MLP output is patched (0-indexed)
    patch_specs : list of dicts with keys
                    subtype  str  "row" | "col" | "box"
                    sidx     int  0-8
                    digit    int  0-8  (0-indexed, i.e. digit 1 = 0)
                    coef     float

    Returns
    -------
    orig_logits, patched_logits : np.ndarray, shape (vocab_size,)
    """
    seq    = session.sequences[puzzle_idx]
    padded = np.full(max_seq_len, int(PAD_TOKEN_BT), dtype=np.int32)
    padded[:len(seq)] = np.array(seq, dtype=np.int32)
    seq_arr = jnp.array(padded)[None]

    orig_logits = np.array(_fwd_clean(seq_arr)[0, seq_pos])

    delta = np.zeros(d_model, dtype=np.float32)
    for spec in patch_specs:
        key = (spec["subtype"], spec["sidx"], spec["digit"])
        if key in probe_vecs[patch_layer] and spec["coef"] != 0.0:
            delta += spec["coef"] * probe_vecs[patch_layer][key]

    if np.any(delta != 0):
        patched_logits = np.array(
            _fwd_patched[patch_layer](seq_arr, seq_pos, jnp.array(delta))[0, seq_pos]
        )
    else:
        patched_logits = orig_logits.copy()

    return orig_logits, patched_logits


# Quick sanity check — first call also triggers JIT compilation for layer 0
_o, _p = run_patched_forward(
    0, 1, 0, [{"subtype": "row", "sidx": 0, "digit": 0, "coef": 5.0}]
)
print(f"Sanity check — logit-diff L2: {np.linalg.norm(_o - _p):.4f}  (should be > 0)")
del _o, _p

In [ ]:
%matplotlib widget

# ─── Figure layout ────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 13))

ax_board   = fig.add_axes([0.02, 0.51, 0.33, 0.44])   # 9x9 board
ax_orig    = fig.add_axes([0.39, 0.74, 0.41, 0.21])   # original logit bars
ax_patched = fig.add_axes([0.39, 0.47, 0.41, 0.21])   # patched logit bars
ax_info    = fig.add_axes([0.02, 0.96, 0.94, 0.03])
ax_info.axis("off")
info_text = ax_info.text(0.01, 0.5, "", fontsize=10, va="center")

ax_token = fig.add_axes([0.80, 0.02, 0.18, 0.93])     # scrollable token list

# Layer radio
ax_layer = fig.add_axes([0.02, 0.27, 0.06, 0.20])
radio_layer = RadioButtons(ax_layer, [str(i) for i in range(n_layers)],
                           active=n_layers - 1)
ax_layer.set_title("Layer", fontsize=9, pad=2)

# Puzzle textbox
ax_puzzle = fig.add_axes([0.12, 0.38, 0.12, 0.04])
txt_puzzle = TextBox(ax_puzzle, "Puzzle: ", initial="0")

# ─── Probe slots (3 rows) ────────────────────────────────────────────────────
# Controls per slot: type textbox | idx textbox | digit textbox | coef slider
# type: "row"/"col"/"box"  |  idx: 0-8  |  digit: 1-9 (human-readable)
SLOT_Y       = [0.20, 0.12, 0.04]
slot_widgets = []

for si, sy in enumerate(SLOT_Y):
    # Slot number label
    ax_lbl = fig.add_axes([0.10, sy + 0.024, 0.025, 0.02])
    ax_lbl.axis("off")
    ax_lbl.text(0.5, 0.5, f"#{si+1}", ha="center", va="center",
                fontsize=9, fontweight="bold")

    ax_st  = fig.add_axes([0.13, sy, 0.07, 0.04])
    txt_st = TextBox(ax_st, "type:", initial="row")

    ax_si  = fig.add_axes([0.22, sy, 0.04, 0.04])
    txt_si = TextBox(ax_si, "idx:", initial="0")

    ax_dg  = fig.add_axes([0.29, sy, 0.04, 0.04])
    txt_dg = TextBox(ax_dg, "dig:", initial="1")

    ax_cf  = fig.add_axes([0.35, sy + 0.007, 0.26, 0.026])
    sl_cf  = Slider(ax_cf, "coef", -10.0, 10.0, valinit=0.0)

    slot_widgets.append({"txt_st": txt_st, "txt_si": txt_si,
                         "txt_dg": txt_dg, "sl_cf":  sl_cf})

# Apply Patch button
ax_btn  = fig.add_axes([0.63, 0.27, 0.12, 0.05])
btn_ref = Button(ax_btn, "Apply Patch", color="#cce8ff", hovercolor="#99d1ff")

# ─── State ────────────────────────────────────────────────────────────────────
state = {
    "puzzle_idx":  0,
    "seq_pos":     0,
    "patch_layer": n_layers - 1,
}
_seq_cache = {"idx": None, "arr": None}

def _get_seq_arr(p_idx: int):
    if _seq_cache["idx"] != p_idx:
        seq    = session.sequences[p_idx]
        padded = np.full(max_seq_len, int(PAD_TOKEN_BT), dtype=np.int32)
        padded[:len(seq)] = np.array(seq, dtype=np.int32)
        _seq_cache["idx"] = p_idx
        _seq_cache["arr"] = jnp.array(padded)[None]
    return _seq_cache["arr"]

def _parse_slot(w: dict):
    try:
        subtype = w["txt_st"].text.strip().lower()
        assert subtype in ("row", "col", "box")
        sidx  = int(w["txt_si"].text.strip())
        assert 0 <= sidx <= 8
        digit = int(w["txt_dg"].text.strip()) - 1   # 1-indexed -> 0-indexed
        assert 0 <= digit <= 8
        coef  = float(w["sl_cf"].val)
        return {"subtype": subtype, "sidx": sidx, "digit": digit, "coef": coef}
    except (AssertionError, ValueError):
        return None

from scripts.analysis.model_readouts import softmax_fill as _softmax_fill

# ─── Render ───────────────────────────────────────────────────────────────────
def render_board():
    p_idx    = state["puzzle_idx"]
    pos      = state["seq_pos"]
    grid_str = build_grid_at_step(session.sequences[p_idx], pos)
    ax_board.clear()
    draw_board(ax_board, grid_str, show_candidates=False)
    seq     = session.sequences[p_idx]
    tok_lbl = get_token_label(int(seq[pos])) if pos < len(seq) else "?"
    ax_board.set_title(f"Puzzle {p_idx} | pos {pos}: {tok_lbl}", fontsize=10)
    fig.canvas.draw_idle()


def render_all(_=None):
    render_board()

    p_idx = state["puzzle_idx"]
    pos   = state["seq_pos"]
    layer = state["patch_layer"]

    seq_arr     = _get_seq_arr(p_idx)
    orig_logits = np.array(_fwd_clean(seq_arr)[0, pos])

    delta       = np.zeros(d_model, dtype=np.float32)
    valid_specs = []
    for w in slot_widgets:
        spec = _parse_slot(w)
        if spec is not None and spec["coef"] != 0.0:
            key = (spec["subtype"], spec["sidx"], spec["digit"])
            if key in probe_vecs[layer]:
                delta += spec["coef"] * probe_vecs[layer][key]
                valid_specs.append(spec)

    if np.any(delta != 0):
        pat_logits = np.array(
            _fwd_patched[layer](seq_arr, pos, jnp.array(delta))[0, pos]
        )
    else:
        pat_logits = orig_logits.copy()

    orig_probs = _softmax_fill(orig_logits)
    pat_probs  = _softmax_fill(pat_logits)

    top_k   = 15
    order   = np.argsort(orig_probs)[::-1][:top_k]
    xlabels = [get_token_label(int(i)) for i in order]
    x_pos   = np.arange(top_k)
    y_max   = max(orig_probs[order].max(), pat_probs[order].max()) * 1.15 + 1e-6

    ax_orig.clear()
    ax_orig.bar(x_pos, orig_probs[order], color="steelblue", alpha=0.85, width=0.7)
    ax_orig.set_xticks(x_pos)
    ax_orig.set_xticklabels(xlabels, rotation=45, ha="right", fontsize=7)
    ax_orig.set_title("Original  (top 15 fill tokens by P)", fontsize=9)
    ax_orig.set_ylim(0, y_max)
    ax_orig.set_ylabel("P(next token)", fontsize=8)

    pat_order   = np.argsort(pat_probs)[::-1][:top_k]
    pat_xlabels = [get_token_label(int(i)) for i in pat_order]
    bar_colors  = [
        "tab:green" if pat_probs[i] > orig_probs[i] + 0.005 else
        "tab:red"   if pat_probs[i] < orig_probs[i] - 0.005 else
        "tab:orange"
        for i in pat_order
    ]
    ax_patched.clear()
    ax_patched.bar(x_pos, pat_probs[pat_order], color=bar_colors, alpha=0.85, width=0.7)
    ax_patched.set_xticks(x_pos)
    ax_patched.set_xticklabels(pat_xlabels, rotation=45, ha="right", fontsize=7)
    ax_patched.set_title(f"Patched (layer {layer+1})  — green=up  red=down", fontsize=9)
    ax_patched.set_ylim(0, y_max)
    ax_patched.set_ylabel("P(next token)", fontsize=8)

    spec_strs = [
        f"#{i+1}: {s['subtype'][0]}{s['sidx']}-d{s['digit']+1} x {s['coef']:.1f}"
        for i, s in enumerate(valid_specs)
    ]
    patch_desc = "  |  ".join(spec_strs) if spec_strs else "no active patch (delta=0)"
    info_text.set_text(
        f"Puzzle {p_idx} | pos {pos} | layer {layer}   ->   {patch_desc}"
    )
    renderer = fig.canvas.get_renderer()
    for ax, fname in [(ax_orig, "patch_orig.png"), (ax_patched, "patch_patched.png")]:
        extent = ax.get_tightbbox(renderer).transformed(fig.dpi_scale_trans.inverted())
        fig.savefig(fname, dpi=200, bbox_inches=extent)
    fig.canvas.draw_idle()


# ─── Token list ───────────────────────────────────────────────────────────────
def _on_token_select(idx: int):
    state["seq_pos"] = idx
    render_board()

token_list = ScrollableTokenList(ax_token, _on_token_select)

def _refresh_token_list():
    labels = decode_sequence_labels(state["puzzle_idx"])
    token_list.set_labels(labels, state["seq_pos"])

# ─── Control callbacks ────────────────────────────────────────────────────────
def _on_puzzle_submit(text):
    try:
        val = int(text.strip()) % n_puzzles
        state["puzzle_idx"] = val
        state["seq_pos"]    = 0
        _seq_cache["idx"]   = None
        _refresh_token_list()
        render_board()
    except ValueError:
        pass

def _on_layer_change(label):
    state["patch_layer"] = int(label)

txt_puzzle.on_submit(_on_puzzle_submit)
radio_layer.on_clicked(_on_layer_change)
btn_ref.on_clicked(render_all)

# ─── Initial draw ─────────────────────────────────────────────────────────────
_refresh_token_list()
render_all()
plt.show()